# Chapter 4 — Kernighan-Lin Algorithm

The previous two chapters used spectral methods — globally informed but computationally expensive ($O(N^3)$).  
**Kernighan-Lin** (1970) takes a different philosophy: start from any partition and iteratively improve it through carefully chosen node swaps.

This chapter covers:
- The classic Kernighan-Lin algorithm
- A degree-normalized variant (our own contribution)
- A direct comparison of all three methods on the same reference graphs

---

## Part 1 — Initialization

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
from scipy.linalg import eigh

In [ ]:
def build_graph(N, M, seed=42, weight=1.0):
    """
    Build a random weighted graph.

    Returns
    -------
    G   : networkx Graph
    A   : adjacency/weight matrix (N x N)
    pos : node layout for plotting
    """
    G = nx.gnm_random_graph(n=N, m=M, seed=seed)
    for u, v in G.edges():
        G[u][v]['weight'] = weight
    pos = nx.spring_layout(G, seed=42)
    A = nx.to_numpy_array(G, weight='weight')
    return G, A, pos


def compute_cut_size(G, set_A, set_B):
    """Total weight of edges crossing between set_A and set_B."""
    return sum(
        G[u][v]['weight']
        for u, v in G.edges()
        if (u in set_A and v in set_B) or (v in set_A and u in set_B)
    )


def plot_partition(G, pos, set_A, set_B, title='', ax=None):
    """Draw graph with nodes colored red (A) or blue (B)."""
    colors = ['#e74c3c' if n in set_A else '#3498db' for n in G.nodes()]
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(6, 5))
    nx.draw(G, pos, with_labels=True, node_color=colors,
            edge_color='gray', node_size=600, font_color='white',
            font_weight='bold', ax=ax)
    ax.set_title(title, fontsize=11)
    if standalone:
        plt.tight_layout()
        plt.show()

---
## Part 2 — Kernighan-Lin Algorithm

### Theory

Given a graph $G = (V, E)$ with $|V| = 2n$, we start with an arbitrary balanced partition $A \cup B = V$, $|A| = |B| = n$.

For each node $v$, define:
$$D(v) = E(v) - I(v)$$
where $E(v)$ is the sum of edge weights from $v$ to the **opposite** set, and $I(v)$ to the **same** set.

The **gain** from swapping a pair $(a, b)$, $a \in A$, $b \in B$ is:
$$g(a, b) = D(a) + D(b) - 2\,c_{ab}$$

**Algorithm (one pass):**
1. Find the pair $(a_1, b_1)$ with maximum gain $g_1$, lock them
2. Update $D$ values for remaining unlocked nodes
3. Repeat for $(a_2, b_2), \ldots, (a_n, b_n)$, recording gains $g_1, \ldots, g_n$
4. Find $k = \arg\max \sum_{i=1}^k g_i$ — apply first $k$ swaps if $G_k > 0$
5. Repeat from new partition until no improvement

In [ ]:
def kernighan_lin(G, seed=None):
    """
    Kernighan-Lin graph partitioning algorithm.

    Parameters
    ----------
    G    : networkx Graph with 'weight' edge attribute
    seed : optional int for reproducible initial partition

    Returns
    -------
    A_final  : set of nodes in partition A
    B_final  : set of nodes in partition B
    history  : list of dicts — cut size and gain per iteration
    """
    nodes = list(G.nodes())
    half = len(nodes) // 2
    A = set(nodes[:half])
    B = set(nodes[half:])

    history = []
    iteration = 0

    while True:
        # Compute D values for all nodes
        D = {}
        for n in G.nodes():
            same_set = A if n in A else B
            opp_set  = B if n in A else A
            I = sum(G[n][nb]['weight'] for nb in G.neighbors(n) if nb in same_set)
            E = sum(G[n][nb]['weight'] for nb in G.neighbors(n) if nb in opp_set)
            D[n] = E - I

        locked = set()
        gains, swaps = [], []

        for _ in range(half):
            best_g, best_a, best_b = float('-inf'), None, None

            for a in A - locked:
                for b in B - locked:
                    c_ab = G[a][b]['weight'] if G.has_edge(a, b) else 0
                    g = D[a] + D[b] - 2 * c_ab
                    if g > best_g:
                        best_g, best_a, best_b = g, a, b

            if best_a is None:
                break

            gains.append(best_g)
            swaps.append((best_a, best_b))
            locked.update([best_a, best_b])

            # Update D values for remaining unlocked nodes
            for n in (A | B) - locked:
                same = A - locked if n in A else B - locked
                opp  = B - locked if n in A else A - locked
                I = sum(G[n][nb]['weight'] for nb in G.neighbors(n) if nb in same)
                E = sum(G[n][nb]['weight'] for nb in G.neighbors(n) if nb in opp)
                D[n] = E - I

        # Find best prefix of swaps
        cumulative = np.cumsum(gains)
        k = int(np.argmax(cumulative)) + 1
        max_gain = cumulative[k - 1]

        cut = compute_cut_size(G, A, B)
        history.append({
            'iteration': iteration + 1,
            'cut_weight': cut,
            'total_gain': max_gain
        })
        iteration += 1

        if max_gain <= 0:
            break

        # Apply the k best swaps
        for a, b in swaps[:k]:
            A.discard(a); A.add(b)
            B.discard(b); B.add(a)

    return A, B, history

### Example 1 — 8 nodes, 10 edges

In [ ]:
G1, A_mat1, pos1 = build_graph(N=8, M=10, seed=30)

initial_A = set(list(G1.nodes())[:4])
initial_B = set(list(G1.nodes())[4:])

kl_A1, kl_B1, history1 = kernighan_lin(G1)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_partition(G1, pos1, initial_A, initial_B,
               title='Initial Partition (sequential split)', ax=axes[0])
plot_partition(G1, pos1, kl_A1, kl_B1,
               title='After Kernighan-Lin', ax=axes[1])
plt.tight_layout()
plt.show()

print(f"Partition A: {sorted(kl_A1)}")
print(f"Partition B: {sorted(kl_B1)}")
print(f"Final cut size: {compute_cut_size(G1, kl_A1, kl_B1):.1f}")
print()
df1 = pd.DataFrame(history1)
print(df1.to_string(index=False))

### Example 2 — 22 nodes, 35 edges

In [ ]:
G2, A_mat2, pos2 = build_graph(N=22, M=35, seed=30)

initial_A2 = set(list(G2.nodes())[:11])
initial_B2 = set(list(G2.nodes())[11:])

kl_A2, kl_B2, history2 = kernighan_lin(G2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_partition(G2, pos2, initial_A2, initial_B2,
               title='Initial Partition', ax=axes[0])
plot_partition(G2, pos2, kl_A2, kl_B2,
               title='After Kernighan-Lin', ax=axes[1])
plt.tight_layout()
plt.show()

print(f"Partition A: {sorted(kl_A2)}")
print(f"Partition B: {sorted(kl_B2)}")
print(f"Final cut size: {compute_cut_size(G2, kl_A2, kl_B2):.1f}")
print()
df2 = pd.DataFrame(history2)
print(df2.to_string(index=False))

---
## Part 3 — Degree-Normalized Variant

### Theory

The standard KL gain treats all edge weights equally regardless of node degree.  
Our variant normalizes by degree, making the gain scale-invariant:

$$d_1(i) = \frac{\sum_{j \in B} A_{ij} - \sum_{\lambda \in C} A_{i\lambda}}{0.001 + \deg(i)}$$

$$d_2(j) = \frac{\sum_{i \in C} A_{ji} - \sum_{\lambda \in B} A_{j\lambda}}{0.001 + \deg(j)}$$

$$dd(i,j) = d_1(i) + d_2(j) - \frac{2A_{ij}}{0.001 + \deg(i)} - \frac{2A_{ji}}{0.001 + \deg(j)}$$

At each iteration, swap the pair $(i_0, j_0)$ with maximum $dd$ value.  
Stop when $\max_{i,j}\, dd(i,j) \leq 0$ or cut cost reaches 0.

The key difference from classic KL: **one swap per iteration** (greedy), and the gain metric accounts for each node's connectivity level.

In [ ]:
def kl_normalized_variant(G):
    """
    Degree-normalized Kernighan-Lin variant.

    Parameters
    ----------
    G : networkx Graph with 'weight' edge attribute

    Returns
    -------
    C_final  : set of nodes in partition C
    B_final  : set of nodes in partition B
    history  : list of dicts — cut cost and max gain per iteration
    """
    nodes = list(G.nodes())
    half = len(nodes) // 2
    C = set(nodes[:half])
    B = set(nodes[half:])

    A_mat = nx.to_numpy_array(G, weight='weight')
    history = []

    e = compute_cut_size(G, C, B)
    e_prev = float('inf')
    iteration = 0

    while e_prev - e > 0:
        iteration += 1
        e_prev = e
        max_gain = float('-inf')
        best_pair = None

        for ci in list(C):
            for bj in list(B):
                deg_ci = float(np.sum(A_mat[ci]))
                deg_bj = float(np.sum(A_mat[bj]))

                d1 = (sum(A_mat[ci][b] for b in B) - sum(A_mat[ci][c] for c in C)) / (0.001 + deg_ci)
                d2 = (sum(A_mat[bj][c] for c in C) - sum(A_mat[bj][b] for b in B)) / (0.001 + deg_bj)
                dd = d1 + d2 \
                   - (2 * A_mat[ci][bj]) / (0.001 + deg_ci) \
                   - (2 * A_mat[bj][ci]) / (0.001 + deg_bj)

                if dd > max_gain:
                    max_gain = dd
                    best_pair = (ci, bj)

        history.append({
            'iteration': iteration,
            'cut_cost': round(e, 2),
            'max_gain': round(max_gain, 4)
        })

        if max_gain <= 0 or e == 0:
            break

        ci, bj = best_pair
        C.discard(ci); C.add(bj)
        B.discard(bj); B.add(ci)
        e = compute_cut_size(G, C, B)

    return C, B, history

### Example 1 — 8 nodes, 10 edges

In [ ]:
var_C1, var_B1, var_history1 = kl_normalized_variant(G1)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_partition(G1, pos1, initial_A, initial_B,
               title='Initial Partition', ax=axes[0])
plot_partition(G1, pos1, var_C1, var_B1,
               title='After Normalized Variant', ax=axes[1])
plt.tight_layout()
plt.show()

print(f"Partition C: {sorted(var_C1)}")
print(f"Partition B: {sorted(var_B1)}")
print(f"Final cut size: {compute_cut_size(G1, var_C1, var_B1):.1f}")
print()
print(pd.DataFrame(var_history1).to_string(index=False))

### Example 2 — 22 nodes, 35 edges

In [ ]:
var_C2, var_B2, var_history2 = kl_normalized_variant(G2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_partition(G2, pos2, initial_A2, initial_B2,
               title='Initial Partition', ax=axes[0])
plot_partition(G2, pos2, var_C2, var_B2,
               title='After Normalized Variant', ax=axes[1])
plt.tight_layout()
plt.show()

print(f"Partition C: {sorted(var_C2)}")
print(f"Partition B: {sorted(var_B2)}")
print(f"Final cut size: {compute_cut_size(G2, var_C2, var_B2):.1f}")
print()
print(pd.DataFrame(var_history2).to_string(index=False))

---
## Part 4 — Full Algorithm Comparison

We now run all three methods — Normalized Cut, Kernighan-Lin, and the normalized variant — on the same reference graphs and compare results directly.

### Reference graph: N=8, M=14, seed=42

In [ ]:
def ncut_partition(G):
    """Single Normalized Cut partition — returns (set_A, set_B, ncut_value)."""
    W = nx.to_numpy_array(G, weight='weight')
    degrees = W.sum(axis=1)
    D_inv_sqrt = np.diag(1.0 / np.sqrt(np.where(degrees > 0, degrees, 1)))
    L_sym = np.eye(len(W)) - D_inv_sqrt @ W @ D_inv_sqrt
    _, eigvecs = eigh(L_sym)
    y1 = D_inv_sqrt @ eigvecs[:, 1]
    set_A = set(np.where(y1 >= 0)[0].tolist())
    set_B = set(np.where(y1 < 0)[0].tolist())
    cut = float(np.sum(W[np.ix_(sorted(set_A), sorted(set_B))]))
    assoc_A = float(np.sum(W[sorted(set_A), :]))
    assoc_B = float(np.sum(W[sorted(set_B), :]))
    ncut = (cut / assoc_A + cut / assoc_B) if assoc_A > 0 and assoc_B > 0 else float('inf')
    return set_A, set_B, round(ncut, 4)


# Reference graph 1: N=8, M=14
Gref1, _, posref1 = build_graph(N=8, M=14, seed=42)

nc_A1, nc_B1, nc_ncut1   = ncut_partition(Gref1)
kl_A1r, kl_B1r, kl_h1r  = kernighan_lin(Gref1)
vr_C1r, vr_B1r, vr_h1r  = kl_normalized_variant(Gref1)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
plot_partition(Gref1, posref1, nc_A1, nc_B1,
               title=f'Normalized Cut  (cut={compute_cut_size(Gref1, nc_A1, nc_B1):.0f})', ax=axes[0])
plot_partition(Gref1, posref1, kl_A1r, kl_B1r,
               title=f'Kernighan-Lin  (cut={compute_cut_size(Gref1, kl_A1r, kl_B1r):.0f})', ax=axes[1])
plot_partition(Gref1, posref1, vr_C1r, vr_B1r,
               title=f'KL Variant  (cut={compute_cut_size(Gref1, vr_C1r, vr_B1r):.0f})', ax=axes[2])
plt.suptitle('Reference Graph: N=8, M=14, seed=42', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### Reference graph: N=12, M=21, seed=42

In [ ]:
Gref2, _, posref2 = build_graph(N=12, M=21, seed=42)

nc_A2, nc_B2, nc_ncut2   = ncut_partition(Gref2)
kl_A2r, kl_B2r, kl_h2r  = kernighan_lin(Gref2)
vr_C2r, vr_B2r, vr_h2r  = kl_normalized_variant(Gref2)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
plot_partition(Gref2, posref2, nc_A2, nc_B2,
               title=f'Normalized Cut  (cut={compute_cut_size(Gref2, nc_A2, nc_B2):.0f})', ax=axes[0])
plot_partition(Gref2, posref2, kl_A2r, kl_B2r,
               title=f'Kernighan-Lin  (cut={compute_cut_size(Gref2, kl_A2r, kl_B2r):.0f})', ax=axes[1])
plot_partition(Gref2, posref2, vr_C2r, vr_B2r,
               title=f'KL Variant  (cut={compute_cut_size(Gref2, vr_C2r, vr_B2r):.0f})', ax=axes[2])
plt.suptitle('Reference Graph: N=12, M=21, seed=42', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### Reference graph: N=21, M=35, seed=30

In [ ]:
Gref3, _, posref3 = build_graph(N=21, M=35, seed=30)

nc_A3, nc_B3, nc_ncut3   = ncut_partition(Gref3)
kl_A3r, kl_B3r, kl_h3r  = kernighan_lin(Gref3)
vr_C3r, vr_B3r, vr_h3r  = kl_normalized_variant(Gref3)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
plot_partition(Gref3, posref3, nc_A3, nc_B3,
               title=f'Normalized Cut  (cut={compute_cut_size(Gref3, nc_A3, nc_B3):.0f})', ax=axes[0])
plot_partition(Gref3, posref3, kl_A3r, kl_B3r,
               title=f'Kernighan-Lin  (cut={compute_cut_size(Gref3, kl_A3r, kl_B3r):.0f})', ax=axes[1])
plot_partition(Gref3, posref3, vr_C3r, vr_B3r,
               title=f'KL Variant  (cut={compute_cut_size(Gref3, vr_C3r, vr_B3r):.0f})', ax=axes[2])
plt.suptitle('Reference Graph: N=21, M=35, seed=30', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### Summary comparison table

In [ ]:
def balance_ratio(A, B):
    return round(min(len(A), len(B)) / max(len(A), len(B)), 2)


rows = []
for graph_label, G, nc_A, nc_B, kl_A, kl_B, vr_C, vr_B in [
    ('N=8,  M=14', Gref1, nc_A1, nc_B1, kl_A1r, kl_B1r, vr_C1r, vr_B1r),
    ('N=12, M=21', Gref2, nc_A2, nc_B2, kl_A2r, kl_B2r, vr_C2r, vr_B2r),
    ('N=21, M=35', Gref3, nc_A3, nc_B3, kl_A3r, kl_B3r, vr_C3r, vr_B3r),
]:
    rows.append({
        'Graph':          graph_label,
        'NCut — cut':     compute_cut_size(G, nc_A, nc_B),
        'NCut — balance': balance_ratio(nc_A, nc_B),
        'KL — cut':       compute_cut_size(G, kl_A, kl_B),
        'KL — balance':   balance_ratio(kl_A, kl_B),
        'Variant — cut':  compute_cut_size(G, vr_C, vr_B),
        'Variant — bal':  balance_ratio(vr_C, vr_B),
    })

df_summary = pd.DataFrame(rows).set_index('Graph')
print(df_summary.to_string())

---
## Summary

| Property | Kernighan-Lin | KL Variant |
|----------|--------------|------------|
| Approach | Iterative local swaps | Iterative local swaps |
| Gain metric | $D(a) + D(b) - 2c_{ab}$ | Degree-normalized $dd(i,j)$ |
| Swaps per iteration | Up to $n/2$ (best prefix) | 1 (greedy) |
| Balance | Enforced ($|A|=|B|$) | Enforced ($|C|=|B|$) |
| Complexity | $O(n^2 \log n)$ per pass | $O(n^2)$ per pass |
| Strength | Fast, reliable improvement | Better on high-degree-variance graphs |

### Across all three methods

| Method | Philosophy | Balance | Cut quality |
|--------|-----------|---------|-------------|
| Spectral Clustering | Global, eigenvector-based | Can be poor | Moderate |
| Normalized Cut | Global, balanced criterion | Good | Good |
| Kernighan-Lin | Local iterative improvement | Enforced | Good |
| KL Variant | Local, degree-aware | Enforced | Competitive |